## 0. Setup — môi trường & tìm data/weight

In [ ]:
# ----- môi trường: tắt warning cho output sạch + tìm data/weight -----
import os, sys, subprocess, zipfile, warnings, logging
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'
logging.getLogger().setLevel(logging.ERROR)

IN_KAGGLE = os.path.isdir('/kaggle/input')
if IN_KAGGLE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'albumentations'], check=False)

def _find_dir(roots, must_contain):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, _ in os.walk(root):
            if all(os.path.exists(os.path.join(dirpath, m)) for m in must_contain):
                return dirpath
    return None

def _find_file(roots, filename):
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            if filename in filenames:
                return os.path.join(dirpath, filename)
    return None

def _extract_zips(roots, dest):
    os.makedirs(dest, exist_ok=True)
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            if os.path.abspath(dirpath).startswith(os.path.abspath(dest)):
                continue
            for fn in filenames:
                if not fn.lower().endswith('.zip'):
                    continue
                marker = os.path.join(dest, '.' + fn + '.done')
                if os.path.exists(marker):
                    continue
                try:
                    with zipfile.ZipFile(os.path.join(dirpath, fn)) as zf:
                        zf.extractall(dest)
                    open(marker, 'w').close()
                    print('extracted:', fn)
                except zipfile.BadZipFile:
                    pass

WEIGHT_NAME = '03_ResNet34_Transformer_OCR.pth'   # baseline A (eval-only)
if IN_KAGGLE:
    INPUT_ROOTS, EXTRACT_DIR, OUT_DIR = ['/kaggle/input'], '/kaggle/working/_data', '/kaggle/working'
else:
    INPUT_ROOTS, EXTRACT_DIR, OUT_DIR = ['..'], os.path.abspath('../_data'), '..'

DATA_ROOT   = _find_dir(INPUT_ROOTS, ['train', 'test_label'])
WEIGHT_PATH = _find_file(INPUT_ROOTS, WEIGHT_NAME)
if not (DATA_ROOT and WEIGHT_PATH):
    _extract_zips(INPUT_ROOTS, EXTRACT_DIR)
    SEARCH = INPUT_ROOTS + [EXTRACT_DIR]
    DATA_ROOT   = DATA_ROOT   or _find_dir(SEARCH, ['train', 'test_label'])
    WEIGHT_PATH = WEIGHT_PATH or _find_file(SEARCH, WEIGHT_NAME)

assert DATA_ROOT, 'Khong tim thay data (can train/ va test_label/).'
assert WEIGHT_PATH, 'Khong tim thay weight 03 (baseline A).'
print('DATA_ROOT :', DATA_ROOT)
print('WEIGHT(A) :', WEIGHT_PATH)
print('OUT_DIR   :', OUT_DIR)

## 1. Import thư viện

In [ ]:
# ----- import thư viện (toàn bộ code model/dataset/train nằm ngay trong notebook) -----
import os, json, math, random
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 2. Mã nguồn đầy đủ (model / dataset / train / eval)
Mỗi cell dưới đây là code thật — đọc & sửa trực tiếp, không nạp từ module ngoài.

### Config — siêu tham số & lựa chọn kiến trúc
`config/config.py`

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple
import torch


@dataclass
class Config:
    path: str = '../data'
    seed: int = 42
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Dataset config
    vocab: str = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    num_classes: int = field(init=False)
    label_len: int = 7
    img_H: int = 32
    img_W: int = 128
    num_frames: int = 5
    layouts: Tuple[str, ...] = ('Brazilian', 'Mercosur')

    # Model config
    embed_dim: int = 512
    ff_dim: int = 512 * 4
    num_layers: int = 3
    num_heads: int = 8
    drop_out: float = 0.1

    # Registry choices (collapse the 7 baseline notebooks into one model)
    backbone: str = 'resnet50'          # resnet34 | resnet50 | convnext_tiny | convnext_base
    decoder: str = 'transformer'        # transformer | bilstm
    fusion_type: str = 'attention'      # mean | max | attention | frame_quality | temporal_transformer
    multi_frame: bool = True            # False -> single-frame baseline
    extractor_pretrained: bool = True
    freeze_extractor: bool = True

    # Super-resolution (multi-task) branch
    use_sr: bool = False
    sr_scale: int = 2
    sr_loss_weight: float = 0.1

    # Layout-classification head (for layout-aware post-processing)
    use_layout_head: bool = False
    layout_loss_weight: float = 0.5

    # Training config
    lr: float = 5e-4
    weight_decay: float = 0.0           # L2 reg; >0 helps warm-start fine-tuning
    batch_size: int = 64
    epochs: int = 30
    log_interval: int = 1
    early_stop_count: int = 5
    warmup_epochs: int = 3
    use_amp: bool = False               # mixed precision (CUDA only); no-op on CPU
    train_loss: List[float] = field(default_factory=list)
    val_loss: List[float] = field(default_factory=list)
    train_acc: List[float] = field(default_factory=list)
    val_acc: List[float] = field(default_factory=list)
    best_model_path: str = 'ResTranOCR.pth'

    def __post_init__(self):
        self.num_classes = len(self.vocab)


### Text codec — mã hoá/giải mã biển số <-> chỉ số lớp
`utils/text_codec.py`

In [ ]:
import torch


def encode_label(label: str, vocab: str) -> torch.Tensor:
    return torch.tensor([vocab.index(c) for c in label.upper()], dtype=torch.long)


def decode_pred(logits: torch.Tensor, vocab: str):
    indices = logits.argmax(dim=2)
    return ["".join(vocab[i] for i in seq.tolist()) for seq in indices]


### Hậu xử lý — logits -> chuỗi biển (rule-based, layout-aware)
`utils/postprocess.py`

In [ ]:
"""Rule-based post-processing for Brazilian / Mercosur plates.

Position grammar (validated on the dataset):
    idx 0,1,2 -> letter (all layouts)
    idx 3     -> digit  (all layouts)
    idx 4     -> digit for Brazilian (LLLDDDD), letter for Mercosur (LLLDLDD)
    idx 5,6   -> digit  (all layouts)

The original notebook rule was layout-agnostic and forced idx 4 toward a digit,
which is wrong for Mercosur. Passing ``layout`` makes idx 4 correct; when the
layout is unknown (blind test without a layout head) idx 4 is left untouched.
"""

# digit -> visually-similar letter
D2C = {'0': 'O', '1': 'I', '2': 'Z', '5': 'S', '6': 'G', '8': 'B', '4': 'A'}
# letter -> visually-similar digit
C2D = {v: k for k, v in D2C.items()}


def _to_letter(ch):
    return D2C.get(ch, ch)


def _to_digit(ch):
    return C2D.get(ch, ch)


def postprocess_rule_base(pred: str, layout: str = None) -> str:
    """Coerce a 7-char prediction to the plate grammar. ``layout`` is optional."""
    chars = list(pred.upper())
    if len(chars) != 7:
        return pred.upper()

    for i in (0, 1, 2):
        chars[i] = _to_letter(chars[i])
    chars[3] = _to_digit(chars[3])

    if layout == 'Brazilian':
        chars[4] = _to_digit(chars[4])
    elif layout == 'Mercosur':
        chars[4] = _to_letter(chars[4])
    else:
        # Unknown layout: keep the legacy soft fix (O -> 0) only.
        if chars[4] == 'O':
            chars[4] = '0'

    for i in (5, 6):
        chars[i] = _to_digit(chars[i])

    return ''.join(chars)


def prediction_from_logits(logits, vocab, layouts=None, layout_logits=None, apply_postprocess=True):
    """Decode logits to strings, optionally applying layout-aware post-processing.

    ``layout_logits`` (B, num_layouts) + ``layouts`` (list of names) enable the
    layout-aware idx-4 rule; otherwise post-processing is layout-agnostic.
    """
    indices = logits.argmax(dim=2).cpu()
    raw = [''.join(vocab[i] for i in seq.tolist()) for seq in indices]
    if not apply_postprocess:
        return raw

    layout_names = [None] * len(raw)
    if layouts is not None and layout_logits is not None:
        pred_layout = layout_logits.argmax(dim=1).cpu().tolist()
        layout_names = [layouts[i] for i in pred_layout]

    return [postprocess_rule_base(p, layout=ln) for p, ln in zip(raw, layout_names)]


### Nạp checkpoint — remap key cũ (02/03) cho khớp model
`utils/checkpoint.py`

In [ ]:
"""Checkpoint loading with backward-compat key remapping.

The registry model renames a few submodules relative to the original notebooks:
    attn_fusion.*       -> fusion.*
    pos_encoder.*       -> decoder.pos_encoder.*
    transformer_layer.* -> decoder.encoder.*
    sequence_model.*     -> decoder.*        (BiLSTM variant)
so that the legacy 02/03 ResNet+Transformer checkpoints load into it.
"""

import torch

_LEGACY_PREFIX_REMAP = [
    ('attn_fusion.', 'fusion.'),
    ('pos_encoder.', 'decoder.pos_encoder.'),
    ('transformer_layer.', 'decoder.encoder.'),
    ('sequence_model.', 'decoder.'),
]


def remap_legacy_state_dict(state_dict):
    new_sd = {}
    for k, v in state_dict.items():
        nk = k
        for old, new in _LEGACY_PREFIX_REMAP:
            if nk.startswith(old):
                nk = new + nk[len(old):]
                break
        new_sd[nk] = v
    return new_sd


def load_checkpoint(model, ckpt_path, map_location='cpu', strict=False):
    """Load a checkpoint into ``model``, remapping legacy keys. Returns the
    (missing_keys, unexpected_keys) tuple from ``load_state_dict``."""
    state = torch.load(ckpt_path, map_location=map_location)
    if isinstance(state, dict):
        for key in ('model_state_dict', 'state_dict'):
            if key in state:
                state = state[key]
                break
    state = remap_legacy_state_dict(state)
    return model.load_state_dict(state, strict=strict)


### Khối dùng chung — STN, positional encoding, Transformer encoder
`models/components.py`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50, ResNet50_Weights


class STNBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.localization = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=5, stride=2, padding=2),
            nn.MaxPool2d(2, 2),
            nn.ReLU(True),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4, 8))
        )
        self.fc_loc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 8, 128),
            nn.ReLU(True),
            nn.Linear(128, 6)
        )
        with torch.no_grad():
            self.fc_loc[-1].weight.zero_()
            self.fc_loc[-1].bias.copy_(
                torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float)
            )
    
    def forward(self, x):
        x = self.localization(x)
        x = self.fc_loc(x)
        x = x.view(-1, 2, 3)
        return x
    

class AttentionFusion(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.score_net = nn.Sequential(
            nn.Conv2d(channels, channels // 8, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 8, 1, kernel_size=1)
        )

    def forward(self, x):
        total_frames, C, H, W = x.size()
        num_frames = 5
        batch_size = total_frames // num_frames

        # Reshape to [Batch_size, Frames, C, H, W]
        x_view = x.view(batch_size, num_frames, C, H, W)

        # Calculate attention scores [Batch_size, Frames, 1, H, W]
        scores = self.score_net(x).view(batch_size, num_frames, 1, H, W)
        weights = F.softmax(scores, dim=1)

        # Weight sim fusion
        fused_features = torch.sum(x_view * weights, dim=1)
        return fused_features


class FeatureExtractor(nn.Module):
    def __init__(self, pretrained=True, out_dim=512, freeze_backbone=False):
        super().__init__()
        weights = ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = resnet50(weights=weights)

        self.conv1   = backbone.conv1
        self.bn1     = backbone.bn1
        self.relu    = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1  = backbone.layer1
        self.layer2  = backbone.layer2
        self.layer3  = backbone.layer3
        self.layer4  = backbone.layer4

        # Modify stride (2,2) -> (2,1) in layer3 và layer4, keep w, shrink h only

        self.layer3[0].conv2.stride = (2, 1)
        self.layer3[0].downsample[0].stride = (2, 1)

        self.layer4[0].conv2.stride = (2, 1)
        self.layer4[0].downsample[0].stride = (2, 1)

        # Project 2048 -> out_dim
        self.proj = nn.Conv2d(2048, out_dim, kernel_size=1)

        if freeze_backbone:
            for name, p in self.named_parameters():
                if 'proj' not in name:
                    p.requires_grad = False

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.proj(x)                              # (B, out_dim, H', W')

        # Collapse height → 1, giữ width làm sequence
        x = F.adaptive_avg_pool2d(x, (1, None))       # (B, out_dim, 1, W')
        return x
    


class pos_encoding(nn.Module):
    def __init__(self, embed_dim, max_length=500):
        super().__init__()
        self.pos_encoding = nn.Parameter(
            (embed_dim ** -0.5) * torch.randn(1, max_length, embed_dim)
        )

    def forward(self, x):
        B, T, C = x.shape
        return x + self.pos_encoding[:, :T, :]
    


class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, ff_dim, num_heads, drop_out):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=drop_out, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim)
        )
        self.layernorm1 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.layernorm2 = nn.LayerNorm(embed_dim, eps=1e-6)
        self.drop1 = nn.Dropout(drop_out)
        self.drop2 = nn.Dropout(drop_out)

    def forward(self, q, k, v):
        q_norm = self.layernorm1(q)
        k_norm = self.layernorm1(k)
        v_norm = self.layernorm1(v)

        attn_out, _ = self.attn(q_norm, k_norm, v_norm)
        attn_out = self.drop1(attn_out)
        out1 = attn_out + q

        out1_norm = self.layernorm2(out1)
        ff_out = self.ff(out1_norm)
        ff_out = self.drop2(ff_out)
        out2 = ff_out + out1

        return out2
    


class TransformerEncoder(nn.Module):
    def __init__(self, embed_dim, ff_dim, num_layers, num_heads, drop_out):
        super().__init__()

        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                embed_dim, ff_dim, num_heads, drop_out
            ) for _ in range(num_layers)
        ])

    def forward(self, x):
        output = x
        for block in self.blocks:
            output = block(output, output, output)
        return output


### Backbone — ResNet/ConvNeXt trích đặc trưng
`models/backbones.py`

In [ ]:
"""Backbone registry — feature extractors that output (B, out_dim, 1, W').

All backbones modify their stride so the height collapses while the width
(which becomes the OCR sequence axis) is preserved. The output is height-pooled
to 1 and projected to ``out_dim`` so every backbone is interchangeable.

Keeping the submodule attribute names (``conv1``/``layer1``/``proj`` for ResNet,
``features``/``proj`` for ConvNeXt) identical to the original notebooks means
checkpoints trained there still load into this registry.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import (
    resnet34, ResNet34_Weights,
    resnet50, ResNet50_Weights,
    convnext_tiny, ConvNeXt_Tiny_Weights,
    convnext_base, ConvNeXt_Base_Weights,
)


class ResNetExtractor(nn.Module):
    """ResNet34/50 backbone with height-collapsing stride modification."""

    def __init__(self, variant="resnet50", pretrained=True, out_dim=512, freeze_backbone=False):
        super().__init__()
        if variant == "resnet50":
            backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
            final_channels = 2048
            is_bottleneck = True
        elif variant == "resnet34":
            backbone = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
            final_channels = 512
            is_bottleneck = False
        else:
            raise ValueError(f"Unknown resnet variant: {variant}")

        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        # Modify stride (2,2) -> (2,1): keep width, shrink height only.
        # Bottleneck (ResNet50) carries the stride on conv2; BasicBlock (ResNet34) on conv1.
        stride_conv = "conv2" if is_bottleneck else "conv1"
        getattr(self.layer3[0], stride_conv).stride = (2, 1)
        self.layer3[0].downsample[0].stride = (2, 1)
        getattr(self.layer4[0], stride_conv).stride = (2, 1)
        self.layer4[0].downsample[0].stride = (2, 1)

        self.proj = nn.Conv2d(final_channels, out_dim, kernel_size=1)

        if freeze_backbone:
            for name, p in self.named_parameters():
                if "proj" not in name:
                    p.requires_grad = False

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.proj(x)
        x = F.adaptive_avg_pool2d(x, (1, None))
        return x


class ConvNeXtExtractor(nn.Module):
    """ConvNeXt tiny/base backbone with height-collapsing stride modification."""

    def __init__(self, variant="convnext_tiny", pretrained=True, out_dim=512, freeze_backbone=False):
        super().__init__()
        if variant == "convnext_tiny":
            backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
            final_channels = 768
        elif variant == "convnext_base":
            backbone = convnext_base(weights=ConvNeXt_Base_Weights.IMAGENET1K_V1 if pretrained else None)
            final_channels = 1024
        else:
            raise ValueError(f"Unknown convnext variant: {variant}")

        self.features = backbone.features

        # Downsampling convs at indices 4 and 6 use kernel=2, stride=2.
        # Make them (2,1)/(2,1) so height halves but width is preserved.
        for ds_idx in [4, 6]:
            conv = self.features[ds_idx][1]
            conv.stride = (2, 1)
            conv.kernel_size = (2, 1)
            with torch.no_grad():
                conv.weight = nn.Parameter(conv.weight[:, :, :, :1].contiguous())

        self.proj = nn.Conv2d(final_channels, out_dim, kernel_size=1)

        if freeze_backbone:
            for name, p in self.named_parameters():
                if "proj" not in name:
                    p.requires_grad = False

    def forward(self, x):
        x = self.features(x)
        x = self.proj(x)
        x = F.adaptive_avg_pool2d(x, (1, None))
        return x


_RESNET = {"resnet34", "resnet50"}
_CONVNEXT = {"convnext_tiny", "convnext_base"}


def build_backbone(name="resnet50", pretrained=True, out_dim=512, freeze_backbone=False):
    """Factory returning a feature extractor for the requested backbone name."""
    if name in _RESNET:
        return ResNetExtractor(name, pretrained, out_dim, freeze_backbone)
    if name in _CONVNEXT:
        return ConvNeXtExtractor(name, pretrained, out_dim, freeze_backbone)
    raise ValueError(
        f"Unknown backbone '{name}'. Available: {sorted(_RESNET | _CONVNEXT)}"
    )


### Fusion 5 frame — mean/max/attention/temporal_transformer
`models/fusion.py`

In [ ]:
"""Multi-frame fusion registry.

Each fusion module takes per-frame feature maps of shape ``(B*F, C, H, W)`` and
returns a single fused map ``(B, C, H, W)``. ``num_frames`` is passed at call
time so the modules are not tied to a fixed frame count.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class MeanFusion(nn.Module):
    """Simplest baseline: average the frames."""

    def forward(self, x, num_frames):
        bf, C, H, W = x.size()
        x = x.view(bf // num_frames, num_frames, C, H, W)
        return x.mean(dim=1)


class MaxFusion(nn.Module):
    """Element-wise max over frames."""

    def forward(self, x, num_frames):
        bf, C, H, W = x.size()
        x = x.view(bf // num_frames, num_frames, C, H, W)
        return x.max(dim=1).values


class AttentionFusion(nn.Module):
    """Original baseline: per-pixel softmax weighting across frames."""

    def __init__(self, channels):
        super().__init__()
        self.score_net = nn.Sequential(
            nn.Conv2d(channels, channels // 8, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 8, 1, kernel_size=1),
        )

    def forward(self, x, num_frames):
        bf, C, H, W = x.size()
        batch_size = bf // num_frames
        x_view = x.view(batch_size, num_frames, C, H, W)
        scores = self.score_net(x).view(batch_size, num_frames, 1, H, W)
        weights = F.softmax(scores, dim=1)
        return torch.sum(x_view * weights, dim=1)


class FrameQualityFusion(nn.Module):
    """One scalar weight per frame (global), i.e. soft best-frame selection.

    A frame's whole feature map is globally pooled to a quality score, then a
    softmax over frames produces per-frame weights shared across all pixels.
    """

    def __init__(self, channels):
        super().__init__()
        self.score_net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // 8),
            nn.ReLU(inplace=True),
            nn.Linear(channels // 8, 1),
        )

    def forward(self, x, num_frames):
        bf, C, H, W = x.size()
        batch_size = bf // num_frames
        x_view = x.view(batch_size, num_frames, C, H, W)
        scores = self.score_net(x).view(batch_size, num_frames, 1, 1, 1)
        weights = F.softmax(scores, dim=1)
        return torch.sum(x_view * weights, dim=1)


class TemporalTransformerFusion(nn.Module):
    """Self-attention across the 5 frames at each spatial location.

    Unlike AttentionFusion (independent per-frame scores), this lets frames
    attend to each other before being pooled, modelling inter-frame relations.
    """

    def __init__(self, channels, num_heads=4, drop_out=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(channels, num_heads, dropout=drop_out, batch_first=True)
        self.norm = nn.LayerNorm(channels)

    def forward(self, x, num_frames):
        bf, C, H, W = x.size()
        batch_size = bf // num_frames
        # (B, F, C, H, W) -> (B*H*W, F, C): each spatial location is a sequence of frames
        x_view = x.view(batch_size, num_frames, C, H, W)
        seq = x_view.permute(0, 3, 4, 1, 2).reshape(batch_size * H * W, num_frames, C)
        seq_norm = self.norm(seq)
        attn_out, _ = self.attn(seq_norm, seq_norm, seq_norm)
        seq = seq + attn_out
        fused = seq.mean(dim=1)  # pool over frames
        fused = fused.view(batch_size, H, W, C).permute(0, 3, 1, 2).contiguous()
        return fused


def build_fusion(name="attention", channels=512, num_heads=4, drop_out=0.1):
    name = name.lower()
    if name == "mean":
        return MeanFusion()
    if name == "max":
        return MaxFusion()
    if name == "attention":
        return AttentionFusion(channels)
    if name == "frame_quality":
        return FrameQualityFusion(channels)
    if name == "temporal_transformer":
        return TemporalTransformerFusion(channels, num_heads, drop_out)
    raise ValueError(
        f"Unknown fusion '{name}'. Available: mean, max, attention, "
        f"frame_quality, temporal_transformer"
    )


### Decoder — Transformer encoder / BiLSTM cho chuỗi ký tự
`models/decoders.py`

In [ ]:
"""Sequence-decoder registry (Transformer / BiLSTM).

Each decoder maps a width-sequence ``(B, T, embed_dim)`` to ``(B, T, embed_dim)``.
The transformer decoder keeps ``pos_encoder`` and ``encoder`` as submodules so
legacy notebook checkpoints (``pos_encoder.*`` / ``transformer_layer.*``) can be
remapped onto it (see ``utils.checkpoint.load_checkpoint``).
"""

import torch.nn as nn


class TransformerDecoder(nn.Module):
    def __init__(self, embed_dim, ff_dim, num_layers, num_heads, drop_out=0.1, max_length=5000):
        super().__init__()
        self.pos_encoder = pos_encoding(embed_dim, max_length=max_length)
        self.encoder = TransformerEncoder(embed_dim, ff_dim, num_layers, num_heads, drop_out)

    def forward(self, x):
        x = self.pos_encoder(x)
        return self.encoder(x)


class BiLSTMDecoder(nn.Module):
    """3-layer bidirectional LSTM; concat of both directions = embed_dim."""

    def __init__(self, embed_dim, num_layers=3, drop_out=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=embed_dim // 2,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=drop_out if num_layers > 1 else 0.0,
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(drop_out)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out)
        out = self.norm(out)
        return out


def build_decoder(name, embed_dim, ff_dim, num_layers, num_heads, drop_out=0.1):
    name = name.lower()
    if name == "transformer":
        return TransformerDecoder(embed_dim, ff_dim, num_layers, num_heads, drop_out)
    if name == "bilstm":
        return BiLSTMDecoder(embed_dim, num_layers, drop_out)
    raise ValueError(f"Unknown decoder '{name}'. Available: transformer, bilstm")


### Super-Resolution module (nhánh đa nhiệm LR->HR)
`models/sr.py`

In [ ]:
"""Lightweight super-resolution / restoration module for the multi-task branch.

A shallow residual CNN with a PixelShuffle upsampler. It enhances each frame
(LR -> SR) before the OCR backbone consumes it. During training the SR output is
supervised by the real HR frames (L1 loss); at inference (blind test, no HR) it
simply runs forward to sharpen the input the OCR sees.
"""

import torch.nn as nn
import torch.nn.functional as F


class _ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return x + self.body(x)


class SRModule(nn.Module):
    def __init__(self, in_channels=3, num_feats=64, num_blocks=4, scale=2):
        super().__init__()
        self.scale = scale
        self.head = nn.Conv2d(in_channels, num_feats, 3, padding=1)
        self.body = nn.Sequential(*[_ResBlock(num_feats) for _ in range(num_blocks)])
        if scale > 1:
            self.upsample = nn.Sequential(
                nn.Conv2d(num_feats, num_feats * scale * scale, 3, padding=1),
                nn.PixelShuffle(scale),
            )
        else:
            self.upsample = nn.Identity()
        self.tail = nn.Conv2d(num_feats, in_channels, 3, padding=1)

    def forward(self, x):
        feat = self.head(x)
        feat = feat + self.body(feat)
        feat = self.upsample(feat)
        out = self.tail(feat)
        # global residual: add bilinearly-upscaled input so the module learns the detail
        base = F.interpolate(x, scale_factor=self.scale, mode="bilinear", align_corners=False) \
            if self.scale > 1 else x
        return out + base


### Model tổng — ResTranOCR (backbone + fusion + decoder + SR) + build_model
`models/restransORC.py`

In [ ]:
"""Configurable multi-frame OCR model (registry-based).

Composition:  STN -> [SR] -> backbone -> fusion -> decoder -> head (+ optional
layout head). Every component is chosen from a registry via :class:`Config`, so
all 7 original notebooks collapse into one model definition.

``forward(x)`` returns the OCR logits ``(B, label_len, num_classes)`` so the
legacy trainer/predictor keep working. ``forward(x, return_aux=True)`` returns a
dict ``{logits, sr, layout}`` used by the unified train/eval harness.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class LayoutHead(nn.Module):
    """Classifies plate layout (e.g. Brazilian vs Mercosur) from fused features."""

    def __init__(self, embed_dim, num_layouts):
        super().__init__()
        self.net = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(embed_dim, embed_dim // 4),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim // 4, num_layouts),
        )

    def forward(self, fused):
        return self.net(fused)


class ResTranOCR(nn.Module):
    def __init__(self, label_len, num_classes, embed_dim, ff_dim, num_layers, num_heads,
                 backbone="resnet50", decoder="transformer", fusion_type="attention",
                 num_frames=5, multi_frame=True,
                 use_sr=False, sr_scale=2, use_layout_head=False, num_layouts=2,
                 extractor_pretrained=True, freeze_extractor=True, drop_out=0.1):
        super().__init__()
        self.label_len = label_len
        self.num_frames = num_frames
        self.multi_frame = multi_frame
        self.use_sr = use_sr
        self.sr_scale = sr_scale
        self.use_layout_head = use_layout_head

        if use_sr:
            self.sr_module = SRModule(in_channels=3, scale=sr_scale)

        self.stn = STNBlock(3)
        self.extractor = build_backbone(
            backbone, pretrained=extractor_pretrained, out_dim=embed_dim,
            freeze_backbone=freeze_extractor,
        )
        if multi_frame:
            self.fusion = build_fusion(fusion_type, channels=embed_dim, num_heads=num_heads, drop_out=drop_out)
        self.decoder = build_decoder(decoder, embed_dim, ff_dim, num_layers, num_heads, drop_out)
        self.head = nn.Linear(embed_dim, num_classes)
        if use_layout_head:
            self.layout_head = LayoutHead(embed_dim, num_layouts)

    def forward(self, x, return_aux=False):
        B, Frames, C, H, W = x.size()
        if not self.multi_frame:
            mid = Frames // 2
            x = x[:, mid:mid + 1]
            Frames = 1
        x_flat = x.reshape(B * Frames, C, H, W)

        sr_out = None
        if self.use_sr:
            sr_out = self.sr_module(x_flat)  # (B*F, C, H*s, W*s)
            ocr_in = F.interpolate(sr_out, size=(H, W), mode="bilinear", align_corners=False)
        else:
            ocr_in = x_flat

        theta = self.stn(ocr_in)
        grid = F.affine_grid(theta, ocr_in.size(), align_corners=False)
        x_aligned = F.grid_sample(ocr_in, grid, align_corners=False)

        features = self.extractor(x_aligned)  # (B*F, embed_dim, 1, W')

        if self.multi_frame:
            fused = self.fusion(features, Frames)  # (B, embed_dim, 1, W')
        else:
            fused = features  # (B, embed_dim, 1, W')

        seq_input = fused.squeeze(2).permute(0, 2, 1)  # (B, W', embed_dim)
        seq_out = self.decoder(seq_input)
        seq_out = seq_out.permute(0, 2, 1)
        seq_out = F.adaptive_avg_pool1d(seq_out, self.label_len)
        seq_out = seq_out.permute(0, 2, 1)
        logits = self.head(seq_out)  # (B, label_len, num_classes)

        if not return_aux:
            return logits

        layout_logits = self.layout_head(fused) if self.use_layout_head else None
        return {"logits": logits, "sr": sr_out, "layout": layout_logits}


def build_model(cfg):
    """Construct a :class:`ResTranOCR` from a :class:`config.config.Config`."""
    return ResTranOCR(
        label_len=cfg.label_len,
        num_classes=cfg.num_classes,
        embed_dim=cfg.embed_dim,
        ff_dim=cfg.ff_dim,
        num_layers=cfg.num_layers,
        num_heads=cfg.num_heads,
        backbone=cfg.backbone,
        decoder=cfg.decoder,
        fusion_type=cfg.fusion_type,
        num_frames=cfg.num_frames,
        multi_frame=cfg.multi_frame,
        use_sr=cfg.use_sr,
        sr_scale=cfg.sr_scale,
        use_layout_head=cfg.use_layout_head,
        num_layouts=len(cfg.layouts),
        extractor_pretrained=cfg.extractor_pretrained,
        freeze_extractor=cfg.freeze_extractor,
        drop_out=cfg.drop_out,
    )


### Augmentation & transform (Albumentations) + target SR
`datasets/transforms.py`

In [ ]:
"""Albumentations pipelines shared by all baselines.

Normalization is fixed at mean=std=0.5 (matching the original training notebooks).
The SR-target transform only resizes + normalizes the clean HR frames so they can
supervise the SR branch at ``scale`` x the OCR input size.
"""

import albumentations as A
from albumentations.pytorch import ToTensorV2

NORM_MEAN = (0.5, 0.5, 0.5)
NORM_STD = (0.5, 0.5, 0.5)


def build_transforms(img_H=32, img_W=128):
    train_transforms = A.Compose([
        A.Resize(height=img_H, width=img_W),
        A.Affine(scale=(0.95, 1.05), translate_percent=(0.05, 0.05), rotate=(-5, 5), fill=128, p=0.5),
        A.Perspective(scale=(0.02, 0.05), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
        A.ChannelShuffle(p=0.3),
        A.CoarseDropout(num_holes_range=(2, 5), hole_height_range=(4, 8), hole_width_range=(4, 8), p=0.3),
        A.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ToTensorV2(),
    ])

    val_test_transforms = A.Compose([
        A.Resize(height=img_H, width=img_W),
        A.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ToTensorV2(),
    ])

    return train_transforms, val_test_transforms


def build_sr_target_transform(img_H=32, img_W=128, scale=2):
    """Clean-HR target for the SR branch (no augmentation, just resize+normalize)."""
    return A.Compose([
        A.Resize(height=img_H * scale, width=img_W * scale),
        A.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ToTensorV2(),
    ])


### Dataset ICPR — đọc track 5 frame LR (+HR cho SR)
`datasets/dataset.py`

In [ ]:
"""Unified ICPR multi-frame dataset (extracted from the baseline notebooks).

Fixes B3: ``track_id`` returned to the predictor is the *original* track id
(e.g. ``track_10005``), not ``track_10005_lr``, so the submission CSV is valid.
The internal ``entry_id`` still carries the ``_lr``/``_hr`` suffix only to keep
the two train-time entries (LR original + HR-degraded) distinct.

Optional extras for the new harness:
- ``return_sr``   -> also yields ``sr_target`` (clean HR frames at scale x size)
- ``return_layout`` -> also yields ``layout`` index (Brazilian/Mercosur)
"""

import json
import os

import albumentations as A
import numpy as np
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset


class ICPRDataSet(Dataset):
    def __init__(self, path, split='train', val_size=0.2, random_state=42, transform=None,
                 sr_transform=None, return_sr=False, return_layout=False,
                 layouts=('Brazilian', 'Mercosur'), vocab=None):
        self.transform = transform
        self.sr_transform = sr_transform
        self.return_sr = return_sr
        self.return_layout = return_layout
        self.layouts = list(layouts)
        self.vocab = vocab
        self.split = split
        self.degradation = self.get_degradation_transforms()

        if split in ['train', 'val']:
            data_path = os.path.join(path, 'train')
            all_tracks = self.get_path(data_path, split='train')
            train_tracks, val_tracks = self.split_tracks(all_tracks, test_size=val_size, random_state=random_state)
            self.tracks = train_tracks if split == 'train' else val_tracks
        elif split == 'test_label':
            self.tracks = self.get_path(os.path.join(path, 'test_label'), split='test_label')
        elif split == 'blind_test':
            self.tracks = self.get_path(os.path.join(path, 'test'), split='test')
        else:
            raise ValueError(f"Unknown split: {split}")

        self.entries = self.build_entries(self.tracks)

        print(f'Split: {split}')
        print(f'Number of tracks: {len(self.tracks)}')
        print(f'Number of samples (entries): {len(self.entries)}')

    def get_path(self, path, split):
        tracks = {}
        track_paths = []

        if split in ('test', 'test_label'):
            for track_name in os.listdir(path):
                track_paths.append(os.path.join(path, track_name))
        else:
            for scenario in ['Scenario-A', 'Scenario-B']:
                for country in ['Brazilian', 'Mercosur']:
                    data_path = os.path.join(path, scenario, country)
                    if not os.path.isdir(data_path):
                        continue
                    for track_name in os.listdir(data_path):
                        track_paths.append(os.path.join(data_path, track_name))

        for track_path in track_paths:
            track_id = os.path.basename(track_path)
            track_lr, track_hr = [], []
            track_label, track_layout = None, None

            for item in os.listdir(track_path):
                item_path = os.path.join(track_path, item)
                if item.lower().startswith('lr'):
                    track_lr.append(item_path)
                elif item.lower().startswith('hr'):
                    track_hr.append(item_path)
                elif item.lower().endswith('.json') and split != 'test':
                    with open(item_path, 'r', encoding='utf-8') as f:
                        annotations = json.load(f)
                    track_label = annotations.get('plate_text').strip()
                    track_layout = annotations.get('plate_layout')

            tracks[track_id] = {
                'track_path': track_path,
                'lr_paths': sorted(track_lr),
                'hr_paths': sorted(track_hr),
                'label': track_label,
                'layout': track_layout,
            }

        return tracks

    def split_tracks(self, tracks, test_size=0.1, random_state=42):
        track_ids = sorted(tracks.keys())
        train_ids, val_ids = train_test_split(track_ids, test_size=test_size, random_state=random_state, shuffle=True)
        train_tracks = {tid: tracks[tid] for tid in train_ids}
        val_tracks = {tid: tracks[tid] for tid in val_ids}
        return train_tracks, val_tracks

    def build_entries(self, tracks):
        entries = []
        for track_id, track in tracks.items():
            # Entry 1: original LR
            entries.append({
                'entry_id': track_id + '_lr',
                'track_id': track_id,
                'paths': track['lr_paths'],
                'hr_paths': track['hr_paths'],
                'use_hr': False,
                'label': track['label'],
                'layout': track['layout'],
            })
            # Entry 2: HR-degraded pseudo-LR (train only)
            if self.split == 'train' and len(track['hr_paths']) > 0:
                entries.append({
                    'entry_id': track_id + '_hr',
                    'track_id': track_id,
                    'paths': track['hr_paths'],
                    'hr_paths': track['hr_paths'],
                    'use_hr': True,
                    'label': track['label'],
                    'layout': track['layout'],
                })
        return entries

    def get_degradation_transforms(self):
        return A.Compose([
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MotionBlur(blur_limit=(3, 5), p=1.0),
            ], p=0.7),
            A.OneOf([
                A.GaussNoise(noise_scale_factor=0.1, p=1.0),
                A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
            ], p=0.7),
            A.ImageCompression(quality_range=(20, 50), p=0.5),
            A.Downscale(scale_range=(0.3, 0.5), p=0.5),
        ])

    def _layout_index(self, layout):
        if layout in self.layouts:
            return self.layouts.index(layout)
        return -1  # unknown (blind test)

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]

        images = []
        raw_images = []
        for img_path in entry['paths']:
            img = np.array(Image.open(img_path).convert('RGB'))
            if entry['use_hr']:
                img = self.degradation(image=img)['image']  # HR -> pseudo-LR
            raw_images.append(img)
            if self.transform is not None:
                img = self.transform(image=img)['image']
            images.append(img)

        sample = {
            'track_id': entry['track_id'],
            'images': images,
            'label': entry['label'],
            'layout': self._layout_index(entry['layout']),
        }

        if self.return_sr and self.sr_transform is not None and len(entry['hr_paths']) > 0:
            sr_targets = []
            for hr_path in entry['hr_paths']:
                hr = np.array(Image.open(hr_path).convert('RGB'))
                sr_targets.append(self.sr_transform(image=hr)['image'])
            sample['sr_target'] = sr_targets

        return sample


### Collate — gộp batch (kèm sr_target/layout khi bật)
`datasets/collate.py`

In [ ]:
"""Collate functions.

The legacy ``collate_fn_train/test/blind_test`` keep the original notebook
signatures. ``collate_harness`` returns a dict and is used by the unified
train/eval harness (carrying optional ``sr_target`` and ``layout``).
"""

import torch


VOCAB = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"


def collate_fn_train(batch):
    images = torch.stack([torch.stack(item['images']) for item in batch])
    targets = torch.stack([encode_label(item['label'], VOCAB) for item in batch])
    return images, targets


def collate_fn_test(batch):
    images = torch.stack([torch.stack(item['images']) for item in batch])
    targets = torch.stack([encode_label(item['label'], VOCAB) for item in batch])
    track_ids = [item['track_id'] for item in batch]
    return images, targets, track_ids


def collate_fn_blind_test(batch):
    images = torch.stack([torch.stack(item['images']) for item in batch])
    track_ids = [item['track_id'] for item in batch]
    return images, track_ids


def collate_harness(batch, vocab=VOCAB):
    out = {
        'images': torch.stack([torch.stack(item['images']) for item in batch]),
        'track_ids': [item['track_id'] for item in batch],
        'layout': torch.tensor([item['layout'] for item in batch], dtype=torch.long),
    }
    if all(item.get('label') is not None for item in batch):
        out['targets'] = torch.stack([encode_label(item['label'], vocab) for item in batch])
    if all('sr_target' in item for item in batch):
        out['sr_target'] = torch.stack([torch.stack(item['sr_target']) for item in batch])
    return out


### Vòng train đa nhiệm (CE + L1 SR) + AMP + cosine + early-stop
`trainers/harness.py`

In [ ]:
"""Unified training harness with optional SR + layout auxiliary losses.

Total loss = CE(ocr) + sr_loss_weight * L1(sr, HR) + layout_loss_weight * CE(layout)

Works for every registry configuration. Uses ``collate_harness`` batches (dicts)
and the model's ``return_aux=True`` path. The legacy ``train_model`` in
``trainer.py`` is left untouched for the original single-loss baselines.
"""

import torch
import torch.nn as nn
from tqdm.auto import tqdm


def compute_accuracy(logits, targets):
    preds = logits.argmax(dim=2)
    correct = (preds == targets).all(dim=1).sum().item()
    return correct / targets.size(0)


def _sr_loss(sr_out, sr_target, batch_size, sr_criterion):
    """Align SR output (B*F_used, C, Hs, Ws) with target (B, F, C, Hs, Ws)."""
    f_used = sr_out.size(0) // batch_size
    f_total = sr_target.size(1)
    if f_used == f_total:
        target = sr_target.reshape(-1, *sr_target.shape[2:])
    else:  # single-frame model used the middle frame
        mid = f_total // 2
        target = sr_target[:, mid:mid + f_used].reshape(-1, *sr_target.shape[2:])
    return sr_criterion(sr_out, target)


def train_harness(model, cfg, train_loader, val_loader, best_model_path, device="cpu"):
    criterion = nn.CrossEntropyLoss()
    sr_criterion = nn.L1Loss()
    layout_criterion = nn.CrossEntropyLoss(ignore_index=-1)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr,
                                 weight_decay=getattr(cfg, "weight_decay", 0.0))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=1e-6)

    use_amp = bool(getattr(cfg, "use_amp", False)) and str(device).startswith("cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_val_acc = 0.0
    break_count = 0
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    warmup = cfg.warmup_epochs
    if warmup > 0:
        for p in model.extractor.parameters():
            p.requires_grad = False

    for epoch in range(cfg.epochs):
        if warmup > 0 and epoch == warmup:
            for p in model.extractor.parameters():
                p.requires_grad = True

        model.train()
        if warmup > 0 and epoch < warmup:
            model.extractor.eval()

        ep_loss, ep_acc = 0.0, 0.0
        for batch in tqdm(train_loader, desc=f"Train e{epoch}", leave=False):
            images = batch["images"].to(device)
            targets = batch["targets"].to(device)
            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(images, return_aux=True)
                logits = out["logits"]
                loss = criterion(logits.permute(0, 2, 1), targets)

                if cfg.use_sr and out["sr"] is not None and "sr_target" in batch:
                    loss = loss + cfg.sr_loss_weight * _sr_loss(
                        out["sr"], batch["sr_target"].to(device), images.size(0), sr_criterion
                    )
                if cfg.use_layout_head and out["layout"] is not None:
                    loss = loss + cfg.layout_loss_weight * layout_criterion(
                        out["layout"], batch["layout"].to(device)
                    )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            ep_loss += loss.item()
            ep_acc += compute_accuracy(logits.detach().cpu(), targets.cpu())

        history["train_loss"].append(ep_loss / len(train_loader))
        history["train_acc"].append(ep_acc / len(train_loader))

        model.eval()
        ep_loss, ep_acc = 0.0, 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Val e{epoch}", leave=False):
                images = batch["images"].to(device)
                targets = batch["targets"].to(device)
                logits = model(images)
                ep_loss += criterion(logits.permute(0, 2, 1), targets).item()
                ep_acc += compute_accuracy(logits.cpu(), targets.cpu())

        history["val_loss"].append(ep_loss / len(val_loader))
        history["val_acc"].append(ep_acc / len(val_loader))
        scheduler.step()

        print(f"| Epoch {epoch:3d} | Train Loss {history['train_loss'][-1]:.4f} "
              f"Acc {history['train_acc'][-1]:.4f} | Val Loss {history['val_loss'][-1]:.4f} "
              f"Acc {history['val_acc'][-1]:.4f} |")

        if history["val_acc"][-1] > best_val_acc:
            break_count = 0
            best_val_acc = history["val_acc"][-1]
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "best_val_acc": best_val_acc, "history": history,
                        "config": vars(cfg)}, best_model_path)
        else:
            break_count += 1
        if break_count >= cfg.early_stop_count:
            print(f"Early stop at epoch {epoch} | best val acc {best_val_acc:.4f}")
            break

    return model, history


### Đánh giá — seq-acc (exact-match) + char-acc + bảng so sánh
`evaluation/eval_harness.py`

In [ ]:
"""Unified evaluation harness on the labelled test set (``test_label``).

Reports, per model/config:
- seq_acc_raw        : exact 7-char match, argmax only
- seq_acc_post       : exact match after rule-based post-processing
                       (layout-aware when the model has a layout head)
- char_acc           : per-character accuracy
- layout_acc         : layout-classification accuracy (if head present)

``run_comparison`` builds several configs, optionally loads checkpoints, and
returns a table so baselines / fusions / SR variants can be compared head-to-head.
"""

import torch
from tqdm.auto import tqdm


@torch.no_grad()
def evaluate_model(model, loader, cfg, device="cpu"):
    model.eval()
    vocab = cfg.vocab
    layouts = list(cfg.layouts)

    n = 0
    seq_raw = seq_post = char_correct = char_total = layout_correct = layout_total = 0

    for batch in tqdm(loader, desc="Eval", leave=False):
        images = batch["images"].to(device)
        targets = batch["targets"]
        gts = ["".join(vocab[i] for i in t.tolist()) for t in targets]

        out = model(images, return_aux=True)
        logits = out["logits"].cpu()

        raw = prediction_from_logits(logits, vocab, apply_postprocess=False)
        if cfg.use_layout_head and out["layout"] is not None:
            post = prediction_from_logits(logits, vocab, layouts=layouts,
                                          layout_logits=out["layout"].cpu(), apply_postprocess=True)
        else:
            post = prediction_from_logits(logits, vocab, apply_postprocess=True)

        for r, p, g in zip(raw, post, gts):
            seq_raw += int(r == g)
            seq_post += int(p == g)
            char_correct += sum(a == b for a, b in zip(p, g))
            char_total += len(g)
        n += len(gts)

        if cfg.use_layout_head and out["layout"] is not None:
            lab = batch["layout"]
            mask = lab >= 0
            if mask.any():
                pred_layout = out["layout"].cpu().argmax(dim=1)
                layout_correct += (pred_layout[mask] == lab[mask]).sum().item()
                layout_total += int(mask.sum())

    return {
        "seq_acc_raw": seq_raw / n if n else 0.0,
        "seq_acc_post": seq_post / n if n else 0.0,
        "char_acc": char_correct / char_total if char_total else 0.0,
        "layout_acc": (layout_correct / layout_total) if layout_total else None,
        "n": n,
    }


def format_table(rows):
    """rows: list of dicts with 'name' + metric keys -> markdown table string."""
    cols = ["name", "seq_acc_raw", "seq_acc_post", "char_acc", "layout_acc", "n"]
    header = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join(["---"] * len(cols)) + " |"
    lines = [header, sep]
    for r in rows:
        vals = []
        for c in cols:
            v = r.get(c)
            if isinstance(v, float):
                vals.append(f"{v:.4f}")
            elif v is None:
                vals.append("-")
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


def run_comparison(configs, loader_builder, device="cpu"):
    """Evaluate several configs and return comparison rows.

    Args:
        configs: list of (name, cfg, checkpoint_path_or_None).
        loader_builder: callable(cfg) -> DataLoader over ``test_label`` using
            ``collate_harness`` (built per-cfg so SR/layout targets match).
    """

    rows = []
    for name, cfg, ckpt in configs:
        model = build_model(cfg).to(device)
        if ckpt is not None:
            missing, unexpected = load_checkpoint(model, ckpt, map_location=device, strict=False)
            print(f"[{name}] loaded {ckpt} | missing={len(missing)} unexpected={len(unexpected)}")
        loader = loader_builder(cfg)
        metrics = evaluate_model(model, loader, cfg, device=device)
        metrics["name"] = name
        rows.append(metrics)
        print(f"[{name}] {metrics}")
    return rows


### Dự đoán blind test -> submission CSV
`predict/predictor.py`

In [ ]:
import os

import torch
from tqdm.auto import tqdm


def predict_blind_test(model, test_loader, vocab: str, device="cpu", save_path="./test_predictions.csv",
                       checkpoint_path=None, apply_postprocess=False, layouts=None):
    if checkpoint_path is not None:
        if not os.path.isfile(checkpoint_path):
            raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

        checkpoint = torch.load(checkpoint_path, map_location=device)
        state_dict = checkpoint.get("model_state_dict", checkpoint)

        try:
            model.load_state_dict(state_dict)
        except RuntimeError:
            if any(k.startswith("module.") for k in state_dict.keys()):
                state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}
            else:
                state_dict = {f"module.{k}": v for k, v in state_dict.items()}
            model.load_state_dict(state_dict)

        print(f"Loaded model weights from: {checkpoint_path}")

    model.eval()
    results = []

    with torch.no_grad():
        use_layout = apply_postprocess and getattr(model, "use_layout_head", False)
        for images, track_ids in tqdm(test_loader, desc="Predict", leave=False):
            images = images.to(device)
            if use_layout:
                out = model(images, return_aux=True)
                preds = prediction_from_logits(
                    out["logits"].cpu(), vocab, layouts=layouts,
                    layout_logits=out["layout"].cpu(), apply_postprocess=True,
                )
            elif apply_postprocess:
                logits = model(images)
                preds = prediction_from_logits(logits.cpu(), vocab, apply_postprocess=True)
            else:
                logits = model(images)
                preds = decode_pred(logits.cpu(), vocab)

            for track_id, pred in zip(track_ids, preds):
                results.append({"track_id": track_id, "plate_text": pred})

    def _sort_key(row):
        track_id = row["track_id"]
        try:
            return int(track_id.split("_")[-1])
        except ValueError:
            return track_id

    results.sort(key=_sort_key)

    with open(save_path, "w", encoding="utf-8") as f:
        f.write("track_id,plate_text\n")
        for row in results:
            f.write(f"{row['track_id']},{row['plate_text']}\n")

    print(f"Saved {len(results)} predictions to: {save_path}")
    return results


## 3. Cấu hình thí nghiệm

In [ ]:
# ===== Cấu hình 2 thí nghiệm: A (nạp 03, eval) vs C (temporal_transformer, train) =====
def build_cfg(name, fusion_type, use_sr):
    cfg = Config(path=DATA_ROOT, backbone='resnet34', decoder='transformer',
                 fusion_type=fusion_type, multi_frame=True, use_sr=use_sr, use_layout_head=False,
                 extractor_pretrained=True, freeze_extractor=False,
                 lr=5e-4, batch_size=64, epochs=50, warmup_epochs=3, early_stop_count=10,
                 use_amp=True, seed=42)
    cfg.device = DEVICE
    cfg.best_model_path = os.path.join(OUT_DIR, f'resnet34_{name}.pth')
    return cfg

EXPERIMENTS = [
    ('A_baseline_03', 'pretrained', 'attention',            False),
    ('C_fusion_only', 'train',      'temporal_transformer', False),
]


## 4. Helpers — dựng loader, eval 2 split

In [ ]:
def make_loader(cfg, split, train=False):
    train_tf, val_tf = build_transforms(cfg.img_H, cfg.img_W)
    tf = train_tf if train else val_tf
    sr_tf = build_sr_target_transform(cfg.img_H, cfg.img_W, cfg.sr_scale) if (train and cfg.use_sr) else None
    ds = ICPRDataSet(cfg.path, split, transform=tf, sr_transform=sr_tf,
                     return_sr=(cfg.use_sr and train), return_layout=cfg.use_layout_head,
                     layouts=cfg.layouts, vocab=cfg.vocab)
    nw = 2 if (IN_KAGGLE and train) else 0
    return DataLoader(ds, batch_size=cfg.batch_size, shuffle=train,
                      collate_fn=partial(collate_harness, vocab=cfg.vocab),
                      num_workers=nw, pin_memory=IN_KAGGLE)

def eval_both(model, cfg):
    res = {}
    for split in ('test_label', 'val'):
        res[split] = evaluate_model(model, make_loader(cfg, split, train=False), cfg, device=cfg.device)
    return res

def run_one(name, mode, fusion_type, use_sr):
    cfg = build_cfg(name, fusion_type, use_sr)
    model = build_model(cfg).to(cfg.device)
    if mode == 'pretrained':
        miss, unexp = load_checkpoint(model, WEIGHT_PATH, map_location=cfg.device, strict=False)
        print(f'[{name}] nap 03 | missing={len(miss)} unexpected={len(unexp)} (eval-only)')
    else:
        n_params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f'[{name}] train tu dau | params={n_params:.1f}M -> {cfg.best_model_path}')
        tr, va = make_loader(cfg, 'train', train=True), make_loader(cfg, 'val', train=False)
        model, _ = train_harness(model, cfg, tr, va, cfg.best_model_path, device=cfg.device)
    return cfg, model, eval_both(model, cfg)


## 5. Chạy — eval A (`03`), train model mới

In [ ]:
results = {}
for name, mode, fusion_type, use_sr in EXPERIMENTS:
    results[name] = run_one(name, mode, fusion_type, use_sr)


## 6. So sánh — Recognition Rate (exact-match) trên `test_label` và `val`
Metric ICPR LR 2026: 1 track đúng chỉ khi **toàn bộ ký tự khớp** (sai 1 ký tự = sai).

In [ ]:
def pct(x): return f'{x*100:.2f}%'
print(f'{"model":<18} | {"fusion":<22} | {"SR":<4} | {"RR test_label":<14} | {"RR val":<10} | char(val)')
print('-'*92)
for name, mode, fusion_type, use_sr in EXPERIMENTS:
    _, _, res = results[name]
    tl, va = res['test_label'], res['val']
    print(f'{name:<18} | {fusion_type:<22} | {str(use_sr):<4} | {pct(tl["seq_acc_post"]):<14} | {pct(va["seq_acc_post"]):<10} | {pct(va["char_acc"])}')


## 7. (tuỳ chọn) Predict blind test -> submission CSV

In [ ]:
best = max(results, key=lambda k: results[k][2]['val']['seq_acc_post'])
bcfg, bmodel, _ = results[best]
print('best (theo val):', best)
_, val_tf = build_transforms(bcfg.img_H, bcfg.img_W)
blind = ICPRDataSet(bcfg.path, 'blind_test', transform=val_tf, vocab=bcfg.vocab)
bl = DataLoader(blind, batch_size=bcfg.batch_size, shuffle=False, collate_fn=collate_fn_blind_test)
csv_path = os.path.join(OUT_DIR, f'submission_{best}.csv')
predict_blind_test(bmodel, bl, bcfg.vocab, device=bcfg.device,
                   save_path=csv_path, apply_postprocess=True, layouts=list(bcfg.layouts))
print('submission ->', csv_path)
